# Project 13: Used Cars Dataset

Complete Data Science workflow, from problem statement to conclusion.

- **Dataset:** Used Cars Dataset (CarDekho)
- **ML task:** Regression (selling price of used cars)
- **Script origin:** `13_used_cars_regression.py` (converted to a notebook)
- **Outputs:** ./outputs/13_used_cars/   (figures, result tables, run_log.txt)
- **Runs on:** local Python 3.9+ or Google Colab, with no changes.
- **Needs:** pandas numpy matplotlib seaborn scikit-learn (+ openpyxl for .xlsx sources)

## 0. Imports and setup

In [ ]:
import os
import re
import sys
import io
import warnings
import zipfile
import urllib.request

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib

# Use a non-interactive backend only when running headless outside notebooks.
if "ipykernel" not in sys.modules and "google.colab" not in sys.modules \
        and not os.environ.get("DISPLAY") and sys.platform.startswith("linux"):
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.feature_selection import SelectFromModel, mutual_info_classif, mutual_info_regression
from sklearn.model_selection import (train_test_split, StratifiedKFold, KFold,
                                     cross_val_score, RandomizedSearchCV)
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
                             classification_report, RocCurveDisplay,
                             mean_absolute_error, mean_squared_error, r2_score,
                             silhouette_score, davies_bouldin_score, calinski_harabasz_score,
                             adjusted_rand_score, normalized_mutual_info_score)
from sklearn.inspection import permutation_importance
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import (RandomForestClassifier, RandomForestRegressor,
                              HistGradientBoostingClassifier, HistGradientBoostingRegressor)
from sklearn.svm import SVC
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA

RS = 42
SLUG = "13_used_cars"
OUT = os.path.join("outputs", SLUG)
os.makedirs(OUT, exist_ok=True)
sns.set_theme(style="whitegrid")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

## Dataset-Specific Configuration And Hooks

Dataset-specific `CONFIG` and helper hooks. Edit `CONFIG` to adapt this notebook to another dataset.

In [ ]:
CONFIG = dict(
    title="Used car selling-price prediction",
    dataset="Used Cars Dataset / CarDekho (301 listings)",
    source="Kaggle 'Vehicle dataset from CarDekho'; CSV copy: https://raw.githubusercontent.com/krishnaik06/Car-Price-Prediction/master/car%20data.csv",
    description="Name, year, present showroom price, kilometres driven, fuel, seller type, transmission, number of previous owners and selling price (lakh INR) of used cars.",
    urls=["https://raw.githubusercontent.com/krishnaik06/Car-Price-Prediction/master/car%20data.csv"],
    problem="Estimate the fair selling price of a used car from its age, usage and original price.",
    objectives=["Clean the listings and handle duplicates and outliers.",
                "Study depreciation: price versus age, kilometres, fuel and transmission.",
                "Create car age, usage-per-year and popularity features.",
                "Compare regression models."],
    tasks=[dict(kind="regression", target="Selling_Price", log_target=True, name="Used car selling price")],
    discussion=["The present (new) price, together with the derived retention proxy, is the strongest predictor, followed by the car's popularity and kilometres driven; fuel type, transmission and seller type also differ in the EDA plots.",
                "Selling price is skewed; the log transformation reduces the influence of a few expensive cars.",
                "Dealer listings are priced higher than individual-seller listings for the same car."],
    future_work=["The dataset is small and old; a larger recent dataset would capture current market conditions.",
                 "Model, trim level and condition are missing and would explain extra variance."],
    conclusion="Used-car prices follow a predictable depreciation pattern anchored on the new price and age; the final model gives accurate estimates for typical cars.",
)

In [ ]:
def create_features(df):
    df["Car_Age"] = int(df["Year"].max()) + 1 - df["Year"]
    df["Kms_per_Year"] = df["Kms_Driven"] / df["Car_Age"].clip(lower=1)
    df["Car_Name_freq"] = df["Car_Name"].map(df["Car_Name"].value_counts())
    df["Price_Retention_Proxy"] = df["Present_Price"] / (df["Car_Age"] + 1)
    return df.drop(columns=["Car_Name", "Year"])

## Generic Helpers

In [ ]:
def hook(name):
    return globals().get(name)

In [ ]:
def section(title):
    print("\n" + "=" * 90 + "\n" + title + "\n" + "=" * 90)

In [ ]:
def savefig(name):
    plt.tight_layout()
    plt.savefig(os.path.join(OUT, name + ".png"), dpi=110, bbox_inches="tight")
    plt.show()
    plt.close("all")

In [ ]:
def is_cat(s):
    return not pd.api.types.is_numeric_dtype(s)

In [ ]:
def all_targets():
    return [t["target"] for t in CONFIG["tasks"] if t.get("target")]

In [ ]:
class Tee:
    def __init__(self, *streams):
        self.streams = streams

    def write(self, s):
        for st in self.streams:
            st.write(s)

    def flush(self):
        for st in self.streams:
            st.flush()

## 1. Problem Statement & Objectives

In [ ]:
def problem_statement():
    section("1. PROBLEM STATEMENT AND PROJECT OBJECTIVES")
    print(CONFIG["title"])
    print("\nProblem statement:\n" + CONFIG["problem"])
    print("\nObjectives:")
    for i, o in enumerate(CONFIG["objectives"], 1):
        print(f"  {i}. {o}")

## 2-3. Dataset Description & Data Collection

In [ ]:
def ask_upload():
    """Last resort: let the user upload the file (Colab) or point to it (local)."""
    try:
        from google.colab import files  # type: ignore
        print("Automatic download failed. Please upload the dataset file.")
        up = files.upload()
        name = next(iter(up))
        reader = hook("read_source")
        if reader and name.lower().endswith((".xlsx", ".xls", ".zip")):
            return reader(name)
        return pd.read_csv(io.BytesIO(up[name]), **CONFIG.get("read_kwargs", {}))
    except ImportError:
        path = os.environ.get("DATA_PATH")
        if path and os.path.exists(path):
            reader = hook("read_source")
            return reader(path) if reader else pd.read_csv(path, **CONFIG.get("read_kwargs", {}))
        raise SystemExit("Could not download the dataset. Download it manually from the URLs in "
                         "CONFIG['urls'] and run:  DATA_PATH=/path/to/file python " + '13_used_cars_regression.py')

In [ ]:
def load_data():
    section("2-3. DATASET DESCRIPTION AND DATA COLLECTION")
    print("Dataset :", CONFIG["dataset"])
    print("Source  :", CONFIG["source"])
    print("About   :", CONFIG["description"])
    df = None
    if hook("custom_load"):
        df = hook("custom_load")()
    else:
        for url in CONFIG["urls"]:
            try:
                reader = hook("read_source")
                df = reader(url) if reader else pd.read_csv(url, **CONFIG.get("read_kwargs", {}))
                print("Loaded from:", url)
                break
            except Exception as e:
                print(f"  ! could not load {url}: {type(e).__name__}: {str(e)[:100]}")
    if df is None and hook("fallback_load"):
        df = hook("fallback_load")()
    if df is None:
        df = ask_upload()
    df.columns = [str(c).strip() for c in df.columns]
    if hook("prepare_raw"):
        df = hook("prepare_raw")(df)
    print(f"\nShape: {df.shape[0]} rows x {df.shape[1]} columns")
    print("\nFirst rows:\n", df.head())
    print("\nColumn types:\n", df.dtypes.astype(str).value_counts().to_string())
    print("\nNumeric summary:\n", df.describe().T.round(3).to_string())
    cat = [c for c in df.columns if is_cat(df[c])]
    if cat:
        print("\nCategorical summary:\n",
              pd.DataFrame({"unique": df[cat].nunique(), "top": df[cat].mode().iloc[0]}).to_string())
    if CONFIG.get("columns_info"):
        print("\nData dictionary:")
        for k, v in CONFIG["columns_info"].items():
            print(f"  - {k}: {v}")
    df.to_csv(os.path.join(OUT, "raw_data_snapshot.csv"), index=False)
    return df

## 4. Data Cleaning & Preprocessing

In [ ]:
def clean_data(df):
    section("4. DATA CLEANING AND PREPROCESSING")
    df = df.copy()
    drop = [c for c in CONFIG.get("drop_cols", []) if c in df.columns]
    if drop:
        df = df.drop(columns=drop)
        print("Dropped identifier / unusable columns:", drop)

    for c in CONFIG.get("force_categorical", []):
        if c in df.columns:
            m = df[c].notna()
            df[c] = df[c].astype(object)
            df.loc[m, c] = df.loc[m, c].astype(str)

    for c in df.columns:  # tidy text
        if is_cat(df[c]):
            df[c] = df[c].astype(object)
            m = df[c].notna()
            df.loc[m, c] = df.loc[m, c].astype(str).str.strip()
            df[c] = df[c].replace({"": np.nan})

    # ---- duplicates
    n0 = len(df)
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"\n[Duplicates] removed {n0 - len(df)} duplicate rows ({n0} -> {len(df)})")

    # ---- constant columns
    const = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
    if const:
        df = df.drop(columns=const)
        print("[Constant columns] dropped:", const)

    # ---- missing values
    miss = df.isna().sum()
    miss = miss[miss > 0]
    print("\n[Missing values]")
    if miss.empty:
        print("  none found")
    else:
        print(pd.DataFrame({"missing": miss, "percent": (miss / len(df) * 100).round(2)}).to_string())
        for c in miss.index:
            if c in all_targets():
                continue
            if miss[c] / len(df) > 0.60:
                print(f"  {c}: >60% missing, kept raw for feature engineering, dropped afterwards")
            elif is_cat(df[c]):
                fill = CONFIG.get("cat_fill", {}).get(c, df[c].mode().iloc[0])
                df[c] = df[c].fillna(fill)
                print(f"  {c}: filled with '{fill}'")
            else:
                df[c] = df[c].fillna(df[c].median())
                print(f"  {c}: filled with median")
    for t in all_targets():
        if t in df.columns and df[t].isna().any():
            df = df.dropna(subset=[t]).reset_index(drop=True)
            print(f"  rows with missing target '{t}' removed")

    # ---- outliers (IQR rule, winsorising)
    skip = set(all_targets()) | set(CONFIG.get("no_outlier_cols", []))
    num = [c for c in df.columns if not is_cat(df[c]) and c not in skip and df[c].nunique() > 10]
    report = []
    before = df[num].copy()
    for c in num:
        q1, q3 = df[c].quantile([0.25, 0.75])
        iqr = q3 - q1
        lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        n_out = int(((df[c] < lo) | (df[c] > hi)).sum())
        if n_out:
            report.append((c, n_out, round(100 * n_out / len(df), 2), round(lo, 3), round(hi, 3)))
            df[c] = df[c].clip(lo, hi)
    print("\n[Outliers] IQR rule (1.5 x IQR); values outside the fences are capped (winsorised)")
    if report:
        rep = pd.DataFrame(report, columns=["column", "n_outliers", "percent", "lower", "upper"])
        print(rep.sort_values("n_outliers", ascending=False).to_string(index=False))
        top = rep.sort_values("n_outliers", ascending=False)["column"].head(6).tolist()
        fig, ax = plt.subplots(1, 2, figsize=(12, 4))
        sns.boxplot(data=(before[top] - before[top].mean()) / before[top].std(), ax=ax[0], orient="h")
        ax[0].set_title("Before capping (standardised)")
        sns.boxplot(data=(df[top] - before[top].mean()) / before[top].std(), ax=ax[1], orient="h")
        ax[1].set_title("After capping (same scale)")
        savefig("04_outliers_before_after")
    else:
        print("  no outliers outside the IQR fences")
    print(f"\nClean data shape: {df.shape}")
    df.to_csv(os.path.join(OUT, "clean_data.csv"), index=False)
    return df

## 5. Exploratory Data Analysis & Visualisation

In [ ]:
def cramers_v(a, b):
    ct = pd.crosstab(a, b)
    if ct.shape[0] < 2 or ct.shape[1] < 2:
        return 0.0
    chi2 = ((ct - np.outer(ct.sum(1), ct.sum(0)) / ct.values.sum()) ** 2 /
            (np.outer(ct.sum(1), ct.sum(0)) / ct.values.sum())).values.sum()
    n = ct.values.sum()
    return float(np.sqrt(chi2 / (n * (min(ct.shape) - 1))))

In [ ]:
def run_eda(df):
    section("5. EXPLORATORY DATA ANALYSIS AND VISUALISATION")
    targets = [t for t in all_targets() if t in df.columns]
    num = [c for c in df.columns if not is_cat(df[c])]
    cat = [c for c in df.columns if is_cat(df[c])]
    print(f"{len(num)} numeric and {len(cat)} categorical columns")

    sk = df[num].skew().sort_values(key=abs, ascending=False).head(8)
    print("\nMost skewed numeric columns:\n", sk.round(2).to_string())

    # 5.1 target distribution
    for t in targets:
        fig, ax = plt.subplots(figsize=(7, 4))
        if is_cat(df[t]) or df[t].nunique() <= 12:
            order = df[t].value_counts().index
            sns.countplot(x=df[t].astype(str), order=[str(o) for o in order], ax=ax)
            plt.xticks(rotation=30, ha="right")
            print(f"\nTarget '{t}' distribution (%):\n", (df[t].value_counts(normalize=True) * 100).round(2).to_string())
        else:
            sns.histplot(df[t], kde=True, ax=ax)
            print(f"\nTarget '{t}': mean={df[t].mean():.3f}, median={df[t].median():.3f}, skew={df[t].skew():.2f}")
        ax.set_title(f"Distribution of target: {t}")
        savefig(f"05_target_{re.sub(r'[^A-Za-z0-9]+', '_', t)}")

    # 5.2 numeric distributions
    cols = [c for c in num if c not in targets][:12]
    if cols:
        n = len(cols)
        r = int(np.ceil(n / 4))
        fig, axes = plt.subplots(r, 4, figsize=(16, 3.2 * r))
        for ax, c in zip(np.array(axes).ravel(), cols):
            sns.histplot(df[c], kde=True, ax=ax)
            ax.set_title(c, fontsize=9)
            ax.set_xlabel("")
        for ax in np.array(axes).ravel()[n:]:
            ax.axis("off")
        savefig("05_numeric_distributions")

    # 5.3 categorical counts
    ccols = [c for c in cat if c not in targets and df[c].nunique() <= 15][:8]
    if ccols:
        r = int(np.ceil(len(ccols) / 4))
        fig, axes = plt.subplots(r, 4, figsize=(16, 3.6 * r))
        for ax, c in zip(np.array(axes).ravel(), ccols):
            vc = df[c].value_counts().head(10)
            sns.barplot(x=vc.values, y=vc.index.astype(str), ax=ax)
            ax.set_title(c, fontsize=9)
        for ax in np.array(axes).ravel()[len(ccols):]:
            ax.axis("off")
        savefig("05_categorical_counts")

    # 5.4 correlation / association
    nn = [c for c in num if df[c].nunique() > 1]
    if len(nn) >= 3:
        if len(nn) > 25 and targets and targets[0] in nn:
            order = df[nn].corr()[targets[0]].abs().sort_values(ascending=False).index[:25]
            nn = list(order)
        elif len(nn) > 25:
            nn = nn[:25]
        corr = df[nn].corr()
        plt.figure(figsize=(max(7, 0.55 * len(nn)), max(6, 0.5 * len(nn))))
        sns.heatmap(corr, annot=len(nn) <= 12, fmt=".2f", cmap="coolwarm", center=0)
        plt.title("Correlation matrix (numeric features)")
        savefig("05_correlation_heatmap")
        pairs = corr.where(np.triu(np.ones(corr.shape), 1).astype(bool)).stack().sort_values(key=abs, ascending=False)
        print("\nStrongest feature-feature correlations:\n", pairs.head(5).round(3).to_string())
    elif len(cat) >= 2 and targets:
        t = targets[0]
        cv = pd.Series({c: cramers_v(df[c], df[t]) for c in cat if c != t}).sort_values(ascending=False)
        plt.figure(figsize=(8, 6))
        sns.barplot(x=cv.values[:15], y=cv.index[:15])
        plt.title(f"Cramer's V association with '{t}'")
        savefig("05_cramers_v")
        print(f"\nStrongest categorical associations with '{t}':\n", cv.head(8).round(3).to_string())

    # 5.5 relationship with the target(s)
    for t in targets:
        tag = re.sub(r"[^A-Za-z0-9]+", "_", t)
        feats_n = [c for c in num if c != t and c not in targets]
        feats_c = [c for c in cat if c != t and df[c].nunique() <= 12]
        clf = is_cat(df[t]) or df[t].nunique() <= 8
        if clf and df[t].nunique() <= 10:
            yy = LabelEncoder().fit_transform(df[t].astype(str))
            if feats_n:
                cr = pd.Series({c: abs(np.corrcoef(df[c], yy)[0, 1]) if df[c].std() > 0 else 0 for c in feats_n})
                top = cr.sort_values(ascending=False).head(6).index
                print(f"\nNumeric features most related to '{t}':\n", cr.sort_values(ascending=False).head(6).round(3).to_string())
                fig, axes = plt.subplots(2, 3, figsize=(15, 7))
                for ax, c in zip(axes.ravel(), top):
                    sns.boxplot(x=df[t].astype(str), y=df[c], ax=ax)
                    ax.set_title(f"{c} by {t}", fontsize=9)
                    ax.tick_params(axis="x", rotation=25)
                for ax in axes.ravel()[len(top):]:
                    ax.axis("off")
                savefig(f"05_numeric_vs_{tag}")
            fc = feats_c[:4]
            if fc:
                fig, axes = plt.subplots(1, len(fc), figsize=(4.5 * len(fc), 4))
                for ax, c in zip(np.atleast_1d(axes), fc):
                    ct = pd.crosstab(df[c], df[t].astype(str), normalize="index") * 100
                    ct.plot(kind="bar", stacked=True, ax=ax, legend=False)
                    ax.set_title(f"{t} share by {c} (%)", fontsize=9)
                    ax.tick_params(axis="x", rotation=30)
                np.atleast_1d(axes)[-1].legend(title=t, bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
                savefig(f"05_categorical_vs_{tag}")
        else:
            if feats_n:
                cr = df[feats_n].corrwith(df[t]).abs().sort_values(ascending=False)
                print(f"\nNumeric features most correlated with '{t}':\n", cr.head(6).round(3).to_string())
                top = cr.head(4).index
                fig, axes = plt.subplots(1, len(top), figsize=(4.3 * len(top), 3.8))
                for ax, c in zip(np.atleast_1d(axes), top):
                    sns.scatterplot(x=df[c], y=df[t], ax=ax, s=14, alpha=0.5)
                    ax.set_title(f"{c} vs {t}", fontsize=9)
                savefig(f"05_numeric_vs_{tag}")
            fc = feats_c[:4]
            if fc:
                fig, axes = plt.subplots(1, len(fc), figsize=(4.5 * len(fc), 4))
                for ax, c in zip(np.atleast_1d(axes), fc):
                    order = df.groupby(c)[t].median().sort_values().index
                    sns.boxplot(x=df[c], y=df[t], order=order, ax=ax)
                    ax.set_title(f"{t} by {c}", fontsize=9)
                    ax.tick_params(axis="x", rotation=35)
                savefig(f"05_categorical_vs_{tag}")

    if hook("eda_extra"):
        hook("eda_extra")(df)
    return df

## 6. Feature Engineering

In [ ]:
def engineer_features(df):
    section("6. FEATURE ENGINEERING (creation, transformation, encoding, scaling)")
    before = set(df.columns)
    df = df.copy()
    if hook("create_features"):
        df = hook("create_features")(df)
    new = [c for c in df.columns if c not in before]
    print("[Feature creation] new columns:", new if new else "none")
    gone = [c for c in before if c not in df.columns]
    if gone:
        print("[Feature creation] columns replaced / removed after parsing:", gone)

    # leftover heavy-missing columns
    for c in [c for c in df.columns if df[c].isna().mean() > 0.60 and c not in all_targets()]:
        df = df.drop(columns=c)
        print(f"[Cleaning] dropped '{c}' (>60% missing)")
    for c in df.columns:
        if c in all_targets() or not df[c].isna().any():
            continue
        df[c] = df[c].fillna(df[c].mode().iloc[0] if is_cat(df[c]) else df[c].median())

    # transformation of skewed numeric features
    skip = set(all_targets()) | set(CONFIG.get("no_transform_cols", []))
    tr = []
    for c in df.columns:
        if is_cat(df[c]) or c in skip or df[c].nunique() <= 10:
            continue
        if df[c].min() >= 0 and df[c].skew() > 1.0:
            sk0 = df[c].skew()
            df[c] = np.log1p(df[c])
            tr.append((c, round(sk0, 2), round(df[c].skew(), 2)))
    print("\n[Feature transformation] log1p applied to skewed features (skew before -> after):")
    print("  " + ", ".join(f"{c}: {a}->{b}" for c, a, b in tr) if tr else "  none needed")

    print("\n[Encoding] categorical features are one-hot encoded inside the model pipeline "
          "(handle_unknown='ignore'); high-cardinality columns (>30 levels) are dropped.")
    print("[Scaling] numeric features are standardised (zero mean, unit variance) inside the pipeline, "
          "which keeps test data out of the scaler fit (no leakage).")
    df.to_csv(os.path.join(OUT, "engineered_data.csv"), index=False)
    print(f"\nEngineered data shape: {df.shape}")
    return df

## 7. Feature Selection & Modelling Helpers

In [ ]:
def make_preprocessor(X):
    num = [c for c in X.columns if not is_cat(X[c])]
    cat = [c for c in X.columns if is_cat(X[c])]
    high = [c for c in cat if X[c].nunique() > 30]
    cat = [c for c in cat if c not in high]
    if high:
        print("  (dropping high-cardinality categorical columns:", high, ")")
    return ColumnTransformer(
        [("num", StandardScaler(), num),
         ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat)],
        remainder="drop", verbose_feature_names_out=False)

In [ ]:
def make_selector(kind):
    est = (RandomForestClassifier if kind == "classification" else RandomForestRegressor)(
        n_estimators=60, max_depth=12, random_state=RS, n_jobs=-1)
    return SelectFromModel(est, threshold="0.25*mean")

In [ ]:
def drop_correlated(X, thr):
    num = [c for c in X.columns if not is_cat(X[c])]
    corr = X[num].corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), 1).astype(bool))
    drop = [c for c in upper.columns if (upper[c] > thr).any()]
    if drop:
        print(f"  [Feature selection - filter] dropped {len(drop)} features with |corr| > {thr}: {drop}")
    return X.drop(columns=drop)

In [ ]:
def build_task_data(df, task):
    excl = set(all_targets()) | set(task.get("drop", []))
    kind = task["kind"]
    if kind == "clustering":
        X = df[task["features"]].copy()
        return X, None, None
    X = df.drop(columns=[c for c in excl if c in df.columns]).copy()
    if CONFIG.get("drop_correlated"):
        X = drop_correlated(X, CONFIG["drop_correlated"])
    y = df[task["target"]]
    names = None
    if kind == "classification":
        if task.get("positive") is not None:
            y = (y.astype(str) == str(task["positive"])).astype(int).values
            names = [task.get("negative_name", "No"), task.get("positive_name", str(task["positive"]))]
        else:
            le = LabelEncoder()
            y = le.fit_transform(y.astype(str))
            names = list(le.classes_)
    else:
        y = y.values.astype(float)
    return X, y, names

In [ ]:
def get_models(kind, n, minority=None, log_target=False):
    if kind == "classification":
        cw = "balanced" if (minority is not None and minority < 0.35) else None
        m = {
            "Baseline (majority class)": DummyClassifier(strategy="most_frequent"),
            "Logistic Regression": LogisticRegression(max_iter=3000, class_weight=cw),
            "K-Nearest Neighbours": KNeighborsClassifier(n_neighbors=7),
            "Decision Tree": DecisionTreeClassifier(max_depth=8, class_weight=cw, random_state=RS),
            "Random Forest": RandomForestClassifier(n_estimators=150 if n > 8000 else 250, class_weight=cw, n_jobs=-1, random_state=RS),
            "Gradient Boosting": HistGradientBoostingClassifier(random_state=RS),
        }
        if n <= 6000:
            m["SVM (RBF)"] = SVC(class_weight=cw, random_state=RS)
        return m
    m = {
        "Baseline (mean predictor)": DummyRegressor(strategy="mean"),
        "Linear Regression": LinearRegression(),
        "Ridge Regression": Ridge(alpha=1.0),
        "Lasso Regression": Lasso(alpha=0.001 if log_target else 0.1, max_iter=20000),
        "K-Nearest Neighbours": KNeighborsRegressor(n_neighbors=7),
        "Decision Tree": DecisionTreeRegressor(max_depth=8, random_state=RS),
        "Random Forest": RandomForestRegressor(n_estimators=250, n_jobs=-1, random_state=RS),
        "Gradient Boosting": HistGradientBoostingRegressor(random_state=RS),
    }
    return m

In [ ]:
GRIDS = {
    "Logistic Regression": {"C": [0.05, 0.3, 1, 3, 10]},
    "K-Nearest Neighbours": {"n_neighbors": [3, 5, 7, 11, 15], "weights": ["uniform", "distance"]},
    "Decision Tree": {"max_depth": [3, 5, 8, 12], "min_samples_leaf": [1, 5, 10, 20]},
    "Random Forest": {"n_estimators": [150, 300], "max_depth": [None, 12, 20], "min_samples_leaf": [1, 2, 4]},
    "Gradient Boosting": {"learning_rate": [0.03, 0.06, 0.1], "max_leaf_nodes": [15, 31], "max_iter": [100, 200, 300]},
    "SVM (RBF)": {"C": [0.5, 1, 3, 10], "gamma": ["scale", 0.02, 0.1]},
    "Ridge Regression": {"alpha": [0.1, 1, 10, 50, 200]},
    "Lasso Regression": {"alpha": [0.0003, 0.001, 0.003, 0.01, 0.05]},
}

In [ ]:
def clf_scores(pipe, Xte, yte, n_classes):
    pred = pipe.predict(Xte)
    avg = "binary" if n_classes == 2 else "weighted"
    auc = np.nan
    try:
        if n_classes == 2:
            s = pipe.predict_proba(Xte)[:, 1] if hasattr(pipe, "predict_proba") else pipe.decision_function(Xte)
            auc = roc_auc_score(yte, s)
        elif hasattr(pipe, "predict_proba"):
            auc = roc_auc_score(yte, pipe.predict_proba(Xte), multi_class="ovr", average="weighted")
    except Exception:
        pass
    return {"Accuracy": accuracy_score(yte, pred),
            "Precision": precision_score(yte, pred, average=avg, zero_division=0),
            "Recall": recall_score(yte, pred, average=avg, zero_division=0),
            "F1": f1_score(yte, pred, average=avg, zero_division=0),
            "ROC-AUC": auc}

In [ ]:
def reg_scores(pipe, Xte, yte):
    pred = pipe.predict(Xte)
    return {"MAE": mean_absolute_error(yte, pred),
            "RMSE": float(np.sqrt(mean_squared_error(yte, pred))),
            "R2": r2_score(yte, pred)}

In [ ]:
def feature_selection_report(pre, sel, Xtr, ytr, kind):
    Z = pre.fit_transform(Xtr)
    names = np.array(pre.get_feature_names_out())
    sel.fit(Z, ytr)
    keep = sel.get_support()
    imp = pd.Series(sel.estimator_.feature_importances_, index=names).sort_values(ascending=False)
    print(f"\n[Feature selection - embedded] RandomForest importance threshold (0.25 x mean) keeps "
          f"{int(keep.sum())} of {len(names)} encoded features")
    print("  Top 10 by importance:", ", ".join(f"{k} ({v:.3f})" for k, v in imp.head(10).items()))
    if (~keep).sum():
        dropped = list(names[~keep])
        print("  Dropped:", dropped[:15], "..." if len(dropped) > 15 else "")
    mi_fn = mutual_info_classif if kind == "classification" else mutual_info_regression
    sub = np.arange(len(Z)) if len(Z) <= 4000 else np.random.RandomState(RS).choice(len(Z), 4000, replace=False)
    mi = pd.Series(mi_fn(Z[sub], np.asarray(ytr)[sub], random_state=RS), index=names).sort_values(ascending=False)
    print("  [filter] Top 5 mutual information:", ", ".join(f"{k} ({v:.3f})" for k, v in mi.head(5).items()))
    plt.figure(figsize=(8, 5))
    imp.head(15)[::-1].plot(kind="barh")
    plt.title("Feature importance used for selection (top 15)")
    return imp

In [ ]:
def run_supervised(df, task):
    kind = task["kind"]
    tag = task.get("name", task["target"])
    slug = re.sub(r"[^A-Za-z0-9]+", "_", tag).strip("_")
    section(f"7-8. MODEL DEVELOPMENT AND EVALUATION: {tag}  [{kind}]")
    X, y, names = build_task_data(df, task)
    log_target = bool(task.get("log_target"))
    print(f"Features used ({X.shape[1]}): {list(X.columns)}")
    n_classes = len(np.unique(y)) if kind == "classification" else None
    minority = (np.bincount(y).min() / len(y)) if kind == "classification" else None
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=RS,
                                          stratify=y if kind == "classification" else None)
    print(f"Train/test split: {len(Xtr)} / {len(Xte)} rows (80/20)")
    n_cv = 3 if len(Xtr) > 10000 else 5
    cv = (StratifiedKFold(n_cv, shuffle=True, random_state=RS) if kind == "classification"
          else KFold(n_cv, shuffle=True, random_state=RS))
    cv_score = ("f1" if n_classes == 2 else "f1_weighted") if kind == "classification" else "r2"
    main_metric = "F1" if kind == "classification" else "R2"
    print(f"Cross-validation: {n_cv}-fold on the training set, scoring = {cv_score}")

    pre0 = make_preprocessor(Xtr)
    feature_selection_report(pre0, make_selector(kind), Xtr, ytr, kind)
    savefig(f"07_feature_importance_{slug}")

    models = get_models(kind, len(Xtr), minority, log_target)
    if kind == "classification" and minority < 0.35:
        print(f"Class imbalance detected (minority = {minority:.1%}): class_weight='balanced' used where supported.")

    def wrap(m):
        return (TransformedTargetRegressor(regressor=m, func=np.log1p, inverse_func=np.expm1)
                if (kind == "regression" and log_target) else m)

    prefix = "model__regressor__" if (kind == "regression" and log_target) else "model__"
    rows, fitted = [], {}
    for name, m in models.items():
        pipe = Pipeline([("prep", clone(pre0)), ("select", make_selector(kind)), ("model", wrap(m))])
        cvs = cross_val_score(pipe, Xtr, ytr, cv=cv, scoring=cv_score)
        pipe.fit(Xtr, ytr)
        sc = clf_scores(pipe, Xte, yte, n_classes) if kind == "classification" else reg_scores(pipe, Xte, yte)
        rows.append({"Model": name, f"CV {main_metric}": cvs.mean(), "CV std": cvs.std(), **sc})
        fitted[name] = pipe
        print(f"  trained {name:<28} CV {main_metric} = {cvs.mean():.4f}   test {main_metric} = {sc[main_metric]:.4f}")

    res = pd.DataFrame(rows)
    cvcol = f"CV {main_metric}"
    cand = res[~res["Model"].str.startswith("Baseline")]
    best_name = cand.loc[cand[cvcol].idxmax(), "Model"]

    # ---- hyper-parameter tuning of the CV-best model
    final_name, final = best_name, fitted[best_name]
    if best_name in GRIDS:
        grid = {prefix + k: v for k, v in GRIDS[best_name].items()}
        base = Pipeline([("prep", clone(pre0)), ("select", make_selector(kind)), ("model", wrap(clone(models[best_name])))])
        n_iter = min(4, int(np.prod([len(v) for v in grid.values()])))
        rs = RandomizedSearchCV(base, grid, n_iter=n_iter, cv=3, scoring=cv_score, random_state=RS, n_jobs=1)
        rs.fit(Xtr, ytr)
        sc = clf_scores(rs.best_estimator_, Xte, yte, n_classes) if kind == "classification" \
            else reg_scores(rs.best_estimator_, Xte, yte)
        tuned = f"{best_name} (tuned)"
        rows.append({"Model": tuned, cvcol: rs.best_score_, "CV std": np.nan, **sc})
        fitted[tuned] = rs.best_estimator_
        print(f"\n[Hyper-parameter tuning] {best_name}: best params = "
              f"{ {k.replace(prefix, ''): v for k, v in rs.best_params_.items()} }")
        print(f"  tuned CV {main_metric} = {rs.best_score_:.4f}, test {main_metric} = {sc[main_metric]:.4f}")
        if rs.best_score_ >= cand[cvcol].max():
            final_name, final = tuned, rs.best_estimator_
        res = pd.DataFrame(rows)

    res = res.sort_values(main_metric, ascending=False).reset_index(drop=True)
    print(f"\n=== MODEL COMPARISON (test set, sorted by {main_metric}) ===")
    print(res.round(4).to_string(index=False))
    res.round(4).to_csv(os.path.join(OUT, f"model_comparison_{slug}.csv"), index=False)
    print(f"\nSelected model (best cross-validation score): {final_name}")

    # ---- comparison plot
    plot = res.melt(id_vars="Model", value_vars=[cvcol, main_metric], var_name="Score", value_name="value")
    plt.figure(figsize=(9, 0.55 * len(res) + 2))
    sns.barplot(data=plot, y="Model", x="value", hue="Score")
    plt.title(f"Model comparison: {main_metric}")
    savefig(f"08_model_comparison_{slug}")

    # ---- detailed evaluation of the final model
    pred = final.predict(Xte)
    detail = {"task": tag, "kind": kind, "best_model": final_name}
    if kind == "classification":
        print("\nClassification report (test set):")
        print(classification_report(yte, pred, target_names=[str(n) for n in names], zero_division=0))
        fig, ax = plt.subplots(figsize=(max(5, 0.8 * n_classes + 3),) * 2)
        ConfusionMatrixDisplay.from_predictions(yte, pred, display_labels=[str(n) for n in names], ax=ax,
                                                normalize="true" if n_classes > 2 else None,
                                                cmap="Blues", xticks_rotation=40)
        ax.set_title(f"Confusion matrix: {final_name}")
        savefig(f"08_confusion_matrix_{slug}")
        if n_classes == 2:
            try:
                fig, ax = plt.subplots(figsize=(5.5, 5))
                RocCurveDisplay.from_estimator(final, Xte, yte, ax=ax)
                ax.plot([0, 1], [0, 1], "k--", alpha=0.4)
                ax.set_title(f"ROC curve: {final_name}")
                savefig(f"08_roc_curve_{slug}")
            except Exception:
                pass
        sc = res[res["Model"] == final_name].iloc[0]
        base = res[res["Model"].str.startswith("Baseline")].iloc[0]
        detail.update({"F1": sc["F1"], "Accuracy": sc["Accuracy"], "ROC-AUC": sc["ROC-AUC"],
                       "baseline_Accuracy": base["Accuracy"], "baseline_F1": base["F1"]})
    else:
        fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
        ax[0].scatter(yte, pred, s=14, alpha=0.5)
        lim = [min(yte.min(), pred.min()), max(yte.max(), pred.max())]
        ax[0].plot(lim, lim, "r--")
        ax[0].set_xlabel("Actual")
        ax[0].set_ylabel("Predicted")
        ax[0].set_title(f"Actual vs predicted: {final_name}")
        sns.histplot(yte - pred, kde=True, ax=ax[1])
        ax[1].set_title("Residuals (actual - predicted)")
        savefig(f"08_regression_diagnostics_{slug}")
        sc = res[res["Model"] == final_name].iloc[0]
        base = res[res["Model"].str.startswith("Baseline")].iloc[0]
        detail.update({"R2": sc["R2"], "MAE": sc["MAE"], "RMSE": sc["RMSE"], "baseline_RMSE": base["RMSE"]})

    # ---- permutation importance on raw input columns
    try:
        idx = np.arange(len(Xte)) if len(Xte) <= 1500 else np.random.RandomState(RS).choice(len(Xte), 1500, replace=False)
        pi = permutation_importance(final, Xte.iloc[idx], np.asarray(yte)[idx], n_repeats=5, random_state=RS,
                                    scoring=cv_score, n_jobs=1)
        s = pd.Series(pi.importances_mean, index=Xte.columns).sort_values(ascending=False)
        print(f"\nPermutation importance (drop in {cv_score} when a feature is shuffled), top 8:")
        print(s.head(8).round(4).to_string())
        plt.figure(figsize=(8, 5))
        s.head(12)[::-1].plot(kind="barh")
        plt.title(f"Permutation importance: {final_name}")
        savefig(f"08_permutation_importance_{slug}")
        detail["top_features"] = ", ".join(s.head(4).index)
    except Exception as e:
        print("(permutation importance skipped:", e, ")")
    return detail

## Clustering

In [ ]:
def run_clustering(df, task):
    tag = task.get("name", "Clustering")
    slug = re.sub(r"[^A-Za-z0-9]+", "_", tag).strip("_")
    section(f"7-8. MODEL DEVELOPMENT AND EVALUATION: {tag}  [clustering]")
    X = df[task["features"]].copy()
    pre = make_preprocessor(X)
    Z = pre.fit_transform(X)
    n = len(Z)
    rng = np.random.RandomState(RS)
    samp = np.arange(n) if n <= task.get("sample", 6000) else rng.choice(n, task.get("sample", 6000), replace=False)
    Zs = Z[samp]
    print(f"Clustering on {X.shape[1]} features, {n} rows (model comparison on {len(samp)} rows)")

    ks = list(range(2, task.get("kmax", 10) + 1))
    inertia, sil = [], []
    for k in ks:
        km = KMeans(k, n_init=10, random_state=RS).fit(Zs)
        inertia.append(km.inertia_)
        sil.append(silhouette_score(Zs, km.labels_))
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(ks, inertia, "o-")
    ax[0].set_title("Elbow method (inertia)")
    ax[0].set_xlabel("k")
    ax[1].plot(ks, sil, "o-", color="green")
    ax[1].set_title("Silhouette score")
    ax[1].set_xlabel("k")
    savefig(f"07_choose_k_{slug}")
    print("Silhouette by k:", {k: round(s, 3) for k, s in zip(ks, sil)})
    best_k = task.get("k") or ks[int(np.argmax(sil))]
    print(f"Chosen k = {best_k}" + (" (fixed in config)" if task.get("k") else " (highest silhouette)"))

    rows, labels_s = [], {}

    def score(name, lab, extra=None):
        mask = lab != -1
        if len(set(lab[mask])) < 2:
            return
        r = {"Model": name, "Clusters": len(set(lab[mask])), "Noise %": round(100 * (~mask).mean(), 1),
             "Silhouette": silhouette_score(Zs[mask], lab[mask]),
             "Davies-Bouldin (low=better)": davies_bouldin_score(Zs[mask], lab[mask]),
             "Calinski-Harabasz": calinski_harabasz_score(Zs[mask], lab[mask])}
        if extra is not None:
            r["ARI vs true"] = adjusted_rand_score(extra[samp][mask], lab[mask])
            r["NMI vs true"] = normalized_mutual_info_score(extra[samp][mask], lab[mask])
        rows.append(r)
        labels_s[name] = lab

    truth = None
    if task.get("compare_labels") and task["compare_labels"] in df.columns:
        truth = LabelEncoder().fit_transform(df[task["compare_labels"]].astype(str))
    score("K-Means", KMeans(best_k, n_init=10, random_state=RS).fit_predict(Zs), truth)
    score("Agglomerative (Ward)", AgglomerativeClustering(best_k).fit_predict(Zs), truth)
    score("Gaussian Mixture", GaussianMixture(best_k, random_state=RS).fit(Zs).predict(Zs), truth)
    best_db = None
    for eps in [0.3, 0.5, 0.7, 0.9, 1.1, 1.4, 1.8, 2.5]:
        lab = DBSCAN(eps=eps, min_samples=max(5, 2 * Z.shape[1])).fit_predict(Zs)
        mask = lab != -1
        if len(set(lab[mask])) >= 2 and (~mask).mean() <= 0.30:
            s = silhouette_score(Zs[mask], lab[mask])
            if best_db is None or s > best_db[0]:
                best_db = (s, eps, lab)
    if best_db:
        score(f"DBSCAN (eps={best_db[1]})", best_db[2], truth)
    else:
        print("DBSCAN: no setting gave >=2 clusters with <=30% noise (reported as not suitable)")
    res = pd.DataFrame(rows).sort_values("Silhouette", ascending=False).reset_index(drop=True)
    print("\n=== CLUSTERING COMPARISON ===")
    print(res.round(4).to_string(index=False))
    res.round(4).to_csv(os.path.join(OUT, f"clustering_comparison_{slug}.csv"), index=False)

    # final K-Means on all rows, profiling
    km = KMeans(best_k, n_init=10, random_state=RS).fit(Z)
    out = df.copy()
    out["Cluster"] = km.labels_
    pcs = PCA(2, random_state=RS).fit(Z)
    P = pcs.transform(Z)
    plt.figure(figsize=(7, 5.5))
    sns.scatterplot(x=P[:, 0], y=P[:, 1], hue=km.labels_, palette="tab10", s=25, alpha=0.7)
    plt.title(f"K-Means clusters (k={best_k}) in PCA space  [{pcs.explained_variance_ratio_.sum():.0%} variance]")
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    savefig(f"08_clusters_pca_{slug}")

    prof_cols = [c for c in task.get("profile_cols", task["features"]) if c in out.columns]
    prof = out.groupby("Cluster")[prof_cols].mean().round(2)
    prof.insert(0, "Size", out["Cluster"].value_counts().sort_index())
    print("\nCluster profiles (feature means):\n", prof.to_string())
    prof.to_csv(os.path.join(OUT, f"cluster_profiles_{slug}.csv"))
    z = (prof[prof_cols] - out[prof_cols].mean()) / out[prof_cols].std()
    print("\nSegment interpretation (z-score vs overall mean; |z| > 0.5 highlighted):")
    for c in z.index:
        hi = [f"high {k}" for k, v in z.loc[c].items() if v > 0.5]
        lo = [f"low {k}" for k, v in z.loc[c].items() if v < -0.5]
        print(f"  Cluster {c} ({int(prof.loc[c, 'Size'])} rows): " + (", ".join(hi + lo) or "close to average on every feature"))
    plt.figure(figsize=(max(7, 1.3 * len(prof_cols)), 4 + 0.4 * best_k))
    sns.heatmap(z, annot=True, fmt=".1f", cmap="coolwarm", center=0)
    plt.title("Cluster profile heatmap (z-scores)")
    savefig(f"08_cluster_profile_heatmap_{slug}")
    out.to_csv(os.path.join(OUT, f"clustered_data_{slug}.csv"), index=False)
    best = res.iloc[0]
    return {"task": tag, "kind": "clustering", "best_model": best["Model"], "k": best_k,
            "Silhouette": best["Silhouette"], "Davies-Bouldin": best["Davies-Bouldin (low=better)"]}

## Model Development Dispatch

In [ ]:
def model_development(df):
    results = []
    for task in CONFIG["tasks"]:
        if task["kind"] == "clustering":
            results.append(run_clustering(df, task))
        else:
            results.append(run_supervised(df, task))
    return results

## 9. Results, Discussion And Conclusion

In [ ]:
def conclusion(results):
    section("9. RESULTS, DISCUSSION AND CONCLUSION")
    print("RESULTS")
    for r in results:
        if r["kind"] == "classification":
            print(f"  * {r['task']}: best model = {r['best_model']}  (test F1 = {r['F1']:.3f}, accuracy = {r['Accuracy']:.3f}"
                  + (f", ROC-AUC = {r['ROC-AUC']:.3f}" if not pd.isna(r.get("ROC-AUC")) else "")
                  + f"; majority-class baseline accuracy = {r['baseline_Accuracy']:.3f})")
        elif r["kind"] == "regression":
            print(f"  * {r['task']}: best model = {r['best_model']}  (test R2 = {r['R2']:.3f}, MAE = {r['MAE']:.3f}, "
                  f"RMSE = {r['RMSE']:.3f}; mean-predictor RMSE = {r['baseline_RMSE']:.3f})")
        elif r["kind"] == "clustering":
            print(f"  * {r['task']}: best method = {r['best_model']} with k = {r['k']} "
                  f"(silhouette = {r['Silhouette']:.3f}, Davies-Bouldin = {r['Davies-Bouldin']:.3f})")
        else:
            print(f"  * {r['task']}: {r.get('summary', '')}")
        if r.get("top_features"):
            print(f"      most influential features: {r['top_features']}")
    print("\nDISCUSSION")
    for line in CONFIG["discussion"]:
        print("  - " + line)
    print("\nLIMITATIONS AND FUTURE WORK")
    for line in CONFIG["future_work"]:
        print("  - " + line)
    print("\nCONCLUSION\n  " + CONFIG["conclusion"])
    pd.DataFrame(results).to_csv(os.path.join(OUT, "results_summary.csv"), index=False)
    print(f"\nAll figures and tables were saved in: {os.path.abspath(OUT)}")

# Run the full workflow
Run the cells below in order. Figures appear inline and are also saved to `outputs/`.

## 1. Problem statement and objectives

In [ ]:
problem_statement()

## 2-3. Dataset description and data collection

In [ ]:
df_raw = load_data()

## 4. Data cleaning and preprocessing

In [ ]:
df_clean = clean_data(df_raw)

## 5. Exploratory data analysis

In [ ]:
df_eda = run_eda(df_clean)

## 6. Feature engineering

In [ ]:
df_fe = engineer_features(df_eda)

## 7-8. Feature selection, modelling and evaluation

In [ ]:
results = hook("custom_model_development")(df_fe) if hook("custom_model_development") else model_development(df_fe)

## 9. Results, discussion and conclusion

In [ ]:
conclusion(results)